In [1]:
import pandas as pd

project_data = pd.read_csv(
    "../data/processed/final_project_data.csv"
)

project_data.head()

,country,year,renewables_share_energy,co2
0,China,2000,5.335688,3643.810
1,China,2001,6.251407,3724.114
2,China,2002,5.921867,4098.181
3,China,2003,5.001339,4835.251
4,China,2004,5.289435,5210.961


In [3]:
import sqlite3
conn = sqlite3.connect(
    "../database/sustainability.db"
)

cursor = conn.cursor()

print("Database Connected")

Database Connected


In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Countries (
    country_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_name TEXT UNIQUE
)
""")

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS RenewableEnergy (
    energy_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_id INTEGER,
    year INTEGER,
    renewables_share_energy REAL,
    FOREIGN KEY(country_id)
    REFERENCES Countries(country_id)
)
""")

In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS CO2Emissions (
    emission_id INTEGER PRIMARY KEY AUTOINCREMENT,
    country_id INTEGER,
    year INTEGER,
    co2 REAL,
    FOREIGN KEY(country_id)
    REFERENCES Countries(country_id)
)
""")

In [7]:
conn.commit()

print("Tables Created Successfully")

Tables Created Successfully


In [8]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('Countries',), ('sqlite_sequence',), ('RenewableEnergy',), ('CO2Emissions',)]


In [10]:
countries = project_data['country'].unique()

for country in countries:
    cursor.execute("""
    INSERT OR IGNORE INTO Countries(country_name)
    VALUES (?)
    """, (country,))

In [11]:
conn.commit()

print("Countries Inserted")

Countries Inserted


In [12]:
countries_df = pd.read_sql_query(
    "SELECT * FROM Countries",
    conn
)

countries_df

,country_id,country_name
0,1,China
1,2,France
2,3,Germany
3,4,India
4,5,Italy
5,6,Spain
6,7,United States


In [13]:
country_lookup = {}

cursor.execute("""
SELECT country_id, country_name
FROM Countries
""")

for row in cursor.fetchall():
    country_lookup[row[1]] = row[0]

country_lookup

{'China': 1,
 'France': 2,
 'Germany': 3,
 'India': 4,
 'Italy': 5,
 'Spain': 6,
 'United States': 7}

In [14]:
for index, row in project_data.iterrows():

    country_id = country_lookup[row['country']]

    cursor.execute("""
    INSERT INTO RenewableEnergy
    (
        country_id,
        year,
        renewables_share_energy
    )
    VALUES (?, ?, ?)
    """,
    (
        country_id,
        int(row['year']),
        float(row['renewables_share_energy'])
    ))

In [15]:
for index, row in project_data.iterrows():

    country_id = country_lookup[row['country']]

    cursor.execute("""
    INSERT INTO CO2Emissions
    (
        country_id,
        year,
        co2
    )
    VALUES (?, ?, ?)
    """,
    (
        country_id,
        int(row['year']),
        float(row['co2'])
    ))

In [16]:
conn.commit()

print("Data Inserted Successfully")

Data Inserted Successfully


In [17]:
renew_df = pd.read_sql_query(
    """
    SELECT *
    FROM RenewableEnergy
    LIMIT 10
    """,
    conn
)

renew_df

,energy_id,country_id,year,renewables_share_energy
0,1,1,2000,5.335688
1,2,1,2001,6.251407
2,3,1,2002,5.921867
3,4,1,2003,5.001339
4,5,1,2004,5.289435
5,6,1,2005,5.238741
6,7,1,2006,5.266756
7,8,1,2007,5.406529
8,9,1,2008,6.872421
9,10,1,2009,6.559855


In [18]:
co2_df = pd.read_sql_query(
    """
    SELECT *
    FROM CO2Emissions
    LIMIT 10
    """,
    conn
)

co2_df

,emission_id,country_id,year,co2
0,1,1,2000,3643.810
1,2,1,2001,3724.114
2,3,1,2002,4098.181
3,4,1,2003,4835.251
4,5,1,2004,5210.961
5,6,1,2005,5881.991
6,7,1,2006,6486.185
7,8,1,2007,6974.663
8,9,1,2008,7492.404
9,10,1,2009,7881.491
